In [9]:
import polars as pl

In [ ]:
import requests
import json
import re

# Convert Google Doc URL to export URL
doc_url = "https://docs.google.com/document/d/1dVsl2f9sL4qr1-XYMfpCwaFFFsLWwUhS0a2kl1yMu7U/edit?tab=t.0"
doc_id = doc_url.split('d/')[1].split('/')[0] 
export_url = f"https://docs.google.com/document/d/{doc_id}/export?format=txt"

# Fetch the document
response = requests.get(export_url)
text_content = response.text

# remove google added characters
data = json.loads(response.text.strip("'").lstrip('\ufeff'))

In [5]:
data = json.loads(response.text.strip("'").lstrip('\ufeff'))

In [15]:
df = pl.DataFrame(data).explode('Games').unnest('Games')

In [ ]:
### Data CLeaning ###
print(df.describe())

# Making the assumption that we can replace NULL with 0

df = df.with_columns([
    # Counting stats: null means zero occurrences
    pl.col('Interceptions').fill_null(0),
    pl.col('Touchdowns').fill_null(0),
    pl.col('Sacks').fill_null(0)
])


shape: (9, 11)
┌────────────┬────────┬───────┬───────────┬───┬────────────┬────────────┬───────────────┬──────────┐
│ statistic  ┆ Player ┆ Game  ┆ Attempts  ┆ … ┆ LongTDPass ┆ Touchdowns ┆ Interceptions ┆ Sacks    │
│ ---        ┆ ---    ┆ ---   ┆ ---       ┆   ┆ ---        ┆ ---        ┆ ---           ┆ ---      │
│ str        ┆ str    ┆ str   ┆ f64       ┆   ┆ f64        ┆ f64        ┆ f64           ┆ f64      │
╞════════════╪════════╪═══════╪═══════════╪═══╪════════════╪════════════╪═══════════════╪══════════╡
│ count      ┆ 60     ┆ 60    ┆ 60.0      ┆ … ┆ 60.0       ┆ 59.0       ┆ 59.0          ┆ 59.0     │
│ null_count ┆ 0      ┆ 0     ┆ 0.0       ┆ … ┆ 0.0        ┆ 1.0        ┆ 1.0           ┆ 1.0      │
│ mean       ┆ null   ┆ null  ┆ 29.033333 ┆ … ┆ 27.533333  ┆ 1.762712   ┆ 0.59322       ┆ 1.559322 │
│ std        ┆ null   ┆ null  ┆ 11.389806 ┆ … ┆ 27.834539  ┆ 1.512465   ┆ 0.832914      ┆ 1.4053   │
│ min        ┆ QB1    ┆ Game1 ┆ 5.0       ┆ … ┆ 0.0        ┆ 0.0        ┆ 0.

#### 1. Which player had the highest single game Completion Percentage?

In [127]:
top_games = (
    df
    .with_columns(
        (pl.col('Completions') / pl.col('Attempts') * 100).alias('CompletionPct')
    )
    .filter(pl.col('Attempts') > 0)
    .sort('CompletionPct', descending=True)
    .head(5)
    .select(['Player', 'Game', 'Completions', 'Attempts', 'CompletionPct'])
)
print(f"Player with the highest single game Completion Percentage: {top_games["Player"][0]}")

Player with the highest single game Completion Percentage: QB5


#### 2. Which player had the lowest single game Yards Per Attempt? 

In [126]:
lowest_syc = (
    df.with_columns((pl.col('Yards')/ pl.col('Attempts')).alias('YardsPerAttempt'))
    .filter(pl.col('Attempts') > 0)
    .sort('YardsPerAttempt')
    .head(5)
    .select(['Player', 'Game', 'Completions', 'Attempts', 'YardsPerAttempt'])
)
print(f"Player with the lowest single game yards per attempt: {lowest_syc["Player"][0]}")

Player with the lowest single game yards per attempt: QB1


#### 3. Which player had the least Passing Yards for the season? 

In [125]:
season_passing_totals = (
    df
    .group_by('Player')
    .agg(
        pl.col('Yards').sum().alias('TotalYards')
    )
    .sort('TotalYards')
    )
print(f"Player with the least passing yards for the season: {season_passing_totals["Player"][0]}")

Player with the least passing yards for the season: QB4


#### 4. Which player had the most Touchdowns for the season? 

In [124]:
season_TD_totals = (
    df
    .group_by("Player")
    .agg(
        pl.col("Touchdowns").sum().alias('TotalTouchdowns')
    )
    .sort('TotalTouchdowns', descending=True)
)
print(f"Player with the most Touchdowns for the season: {season_TD_totals['Player'][0]}")

Player with the most Touchdowns for the season: QB2


In [47]:
season_TD_totals[0,0]

'QB2'

#### 5. List the player names by their Season Completion Percentage in descending order. List the names in order, separated by a comma with no spaces like this: QB1,QB2,QB3,QB4,QB5 

In [ ]:
player_comp_percent = (
    df
    .group_by("Player")
    .agg([
        pl.col("Completions").sum().alias('TotalCompletions'),
        pl.col("Attempts").sum().alias('TotalAttempts')
    ])
    .with_columns(
        (pl.col('TotalCompletions') / pl.col('TotalAttempts') * 100).alias('CompletionPct')
    )
    .sort('CompletionPct', descending=True)
)

player_list = ','.join(player_comp_percent['Player'].to_list())
print(f"Completion % ranking: {player_list}")

5. Completion % ranking: QB5,QB2,QB3,QB1,QB4


In [104]:
def calc_passer_rating(df: pl.DataFrame):
    qb_stats = (
    df
    .with_columns([
        # Completion percentage component
        ((pl.col('Completions') / pl.col('Attempts') - 0.3) * 5).alias('a_component'),
        ((pl.col('Yards')/ pl.col('Attempts') - 3.0) * 0.25).alias('b_component'),
        ((pl.col("Touchdowns") / pl.col('Attempts'))*20.0).alias("c_component"),
        (2.375-(pl.col("Interceptions")/pl.col("Attempts") * 25)).alias("d_component")
    ])
    .with_columns([
        # Cap each component between 0 and 2.375
        pl.col('a_component').clip(0, 2.375).alias('a_final'),
        pl.col('b_component').clip(0, 2.375).alias('b_final'),
        pl.col('c_component').clip(0, 2.375).alias('c_final'),
        pl.col('d_component').clip(0, 2.375).alias('d_final')
    ])
    .with_columns(
        # Calculate final passer rating
        ((pl.col('a_final') + pl.col('b_final') + 
          pl.col('c_final') + pl.col('d_final')) / 6 * 100).alias('PasserRating')
    )
    .sort(pl.col('PasserRating'), descending=True)
    .drop(['a_component', 'b_component', 'c_component', 'd_component',
           'a_final', 'b_final', 'c_final', 'd_final'])
)

    return qb_stats.sort('PasserRating', descending=True)


#### 6. Which player had the highest single game Passer Rating?

In [105]:
qb_stats = calc_passer_rating(df)

top_qb = qb_stats.head(1).select('Player').item()
passer_rating = qb_stats.head(1).select('PasserRating').item()

print(f"{top_qb} achieved the highest single game Passer Rating.")


QB1 achieved the highest single game Passer Rating.


#### What was the value of the highest single game Passer Rating?

In [93]:
print(f"Highest single game Passer Rating for the season was {passer_rating:.1f}")

Highest single game Passer Rating for the season was 158.3


In [81]:
qb_stats

Player,Game,PasserRating
str,str,f64
"""QB1""","""Game1""",158.333333
"""QB5""","""Game7""",158.143939
"""QB2""","""Game5""",156.25
"""QB3""","""Game8""",154.166667
"""QB5""","""Game6""",144.791667
…,…,…
"""QB1""","""Game4""",48.263889
"""QB1""","""Game12""",47.916667
"""QB4""","""Game1""",41.856061


#### 8. Which player had the lowest single game Passer Rating? 

In [94]:
lowest_passer_rating_player = (qb_stats.tail(1).select(["Player"]).item())
lowest_passer_rating = (qb_stats.tail(1).select(["PasserRating"]).item())

print(f"{lowest_passer_rating_player} had the lowest single game Passer Rating")

QB3 had the lowest single game Passer Rating


#### 9. What was the value of the lowest single game Passer Rating?

In [95]:
print(f"The value of the lowest single game Passer Rating was: {lowest_passer_rating:.1f}")

The value of the lowest single game Passer Rating was: 27.6


#### 10. Which player had the highest season Passer Rating? 

In [107]:
season_totals = (
    df
    .group_by('Player')
    .agg([
        pl.col('Attempts').sum(),     
        pl.col('Completions').sum(),  
        pl.col('Yards').sum(),        
        pl.col('Touchdowns').sum(),   
        pl.col('Interceptions').sum() 
    ])
)
seasonal_passer_rating = calc_passer_rating(season_totals)

player = (seasonal_passer_rating
          .sort(pl.col("PasserRating"), descending=True)
          .head(1).select(["Player"])
          .item())

print(f"{player} had the highest season Passer Rating")

QB5 had the highest season Passer Rating


#### 11. Which player had the highest Passer Rating through Game1, Game2 and Game3?

In [108]:
game_subset = df.sort(pl.col("Game")).head(3)

In [112]:
game_subset = (
    df
    .filter(pl.col('Game').is_in(['Game1', 'Game2', 'Game3']))
    .group_by('Player')
    .agg([
        pl.col('Attempts').sum(),
        pl.col('Completions').sum(),
        pl.col('Yards').sum(),
        pl.col('Touchdowns').sum(),
        pl.col('Interceptions').sum()
    ])
)
game_subset_stats = calc_passer_rating(game_subset)

player = (game_subset_stats
          .sort(pl.col("PasserRating"), descending=True)
          .head(1).select(["Player"])
          .item())

print(f"{player} had the highest season Passer Rating")


QB1 had the highest season Passer Rating


In [111]:
qb_stats

Player,Attempts,Completions,Yards,Touchdowns,Interceptions,PasserRating
str,i64,i64,i64,i64,i64,f64
"""QB1""",84,58,755,7,0,124.85119
"""QB2""",83,57,763,6,0,121.711847
"""QB5""",52,31,605,6,3,114.663462
"""QB3""",107,69,748,1,1,84.170561
"""QB4""",49,24,212,3,2,64.328231


#### 12. Excluding each player’s highest and lowest single game Passer Rating, which player had the highest Passer Rating for the season?

In [ ]:
df = calc_passer_rating(df)
df_filtered = (
    df
    .with_columns(
        pl.col('PasserRating').rank().over('Player').alias('rank_asc'),
        pl.col('PasserRating').rank(descending=True).over('Player').alias('rank_desc')
    )
    .filter((pl.col('rank_asc') > 1) & (pl.col('rank_desc') > 1))
)
season_totals_filtered = (
    df_filtered
    .group_by('Player')
    .agg([
        pl.col('Attempts').sum(),     
        pl.col('Completions').sum(),  
        pl.col('Yards').sum(),        
        pl.col('Touchdowns').sum(),   
        pl.col('Interceptions').sum(),
    ])
)
qb_stats_filtered = calc_passer_rating(season_totals_filtered)
player =  (qb_stats_filtered
          .sort(pl.col("PasserRating"), descending=True)
          .head(1).select(["Player"])
          .item())

print(f"Exluding highest and lowest single game Passer Rating, {player} had the highest Passer Rating for the season")


Exluding highest and lowest single game Passer Rating, QB2 had the highest Passer Rating for the season
